# 03: Oil and Cross-species Adulteration

Food fraud detection and quantification.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, precision_recall_curve, average_precision_score
from fishy.analysis.xai import GradCAM, ModelWrapper
from lime.lime_tabular import LimeTabularExplainer
from fishy._core.utils import get_device
from fishy import TrainingConfig, run_unified_training, display_final_summary


In [ ]:
# Train Oil Model
config = TrainingConfig(model="rf", dataset="oil")
results = run_unified_training(config)
display_final_summary(results)


### Ordinal Adulteration Gradient
For the oil dataset, we visualize the correlation between true and predicted adulteration percentages.

In [ ]:
oil_map = {0: 50, 1: 25, 2: 10, 3: 5, 4: 1, 5: 0.1, 6: 0}
preds = results["predictions"]
y_true = np.array([oil_map[idx] for idx in preds["labels"]])
y_pred = np.array([oil_map[idx] for idx in preds["preds"]])
fig_oil = px.scatter(x=y_true, y=y_pred, trendline="ols", 
                     labels={"x": "True Adulteration %", "y": "Predicted Adulteration %"}, 
                     title="Oil Adulteration: True vs Predicted Gradient")
fig_oil.show()


## Results & Interpretability

We now visualize the model's performance and use Explainable AI (XAI) techniques to understand its decision-making process.

In [ ]:
# 1. Training Progress (Loss and Accuracy Curves)
if "epoch_metrics" in results:
    history = results["epoch_metrics"]
    fig_loss = px.line(pd.DataFrame({"Epoch": range(1, len(history["train_losses"]) + 1), 
                                   "Train Loss": history["train_losses"], 
                                   "Val Loss": history["val_losses"]}), 
                       x="Epoch", y=["Train Loss", "Val Loss"], title="Training & Validation Loss")
    fig_loss.show()
    
    val_accs = [m.get("balanced_accuracy", 0) for m in history["val_metrics"]]
    fig_acc = px.line(y=val_accs, title="Validation Balanced Accuracy per Epoch", labels={"x": "Epoch", "y": "Accuracy"})
    fig_acc.show()


In [ ]:
# 2. Confusion Matrix & Class Metrics
preds = results.get("predictions")
class_names = results.get("class_names")
if preds and class_names:
    y_true, y_pred = preds["labels"], preds["preds"]
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig_cm = px.imshow(cm, x=class_names, y=class_names, text_auto=True, 
                       color_continuous_scale="Blues", title="Confusion Matrix")
    fig_cm.show()
    
    # Class-wise Precision, Recall, F1
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(class_names)))
    metrics_df = pd.DataFrame({"Class": class_names, "Precision": prec, "Recall": rec, "F1-Score": f1})
    fig_metrics = px.bar(metrics_df, x="Class", y=["Precision", "Recall", "F1-Score"], 
                         barmode="group", title="Class-Specific Performance")
    fig_metrics.show()


In [ ]:
# 3. Precision-Recall Curves
y_probs = preds.get("probs")
if y_probs is not None:
    fig_pr = go.Figure()
    for i in range(len(class_names)):
        precision, recall, _ = precision_recall_curve(y_true == i, y_probs[:, i])
        avg_prec = average_precision_score(y_true == i, y_probs[:, i])
        fig_pr.add_trace(go.Scatter(x=recall, y=precision, name=f"{class_names[i]} (AP={avg_prec:.2f})", mode="lines"))
    fig_pr.update_layout(title="Precision-Recall Curves", xaxis_title="Recall", yaxis_title="Precision", template="plotly_white")
    fig_pr.show()


In [ ]:
# 4. Interpretability (LIME)
model = results.get("model")
dm = results.get("data_module")
if model and dm:
    X, y = dm.get_numpy_data(labels_as_indices=True)
    wrapper = ModelWrapper(model, str(get_device()))
    feature_names = [f"{c}" for c in dm.get_filtered_dataframe().columns if c not in ["Class Name", "m/z"]]
    
    explainer = LimeTabularExplainer(X, feature_names=feature_names, class_names=class_names, discretize_continuous=True)
    # Explain the first sample
    exp = explainer.explain_instance(X[0], wrapper.predict_proba, num_features=10)
    
    exp_list = exp.as_list()
    fig_lime = px.bar(x=[x[1] for x in exp_list], y=[x[0] for x in exp_list], orientation="h",
                      color=[x[1] for x in exp_list], color_continuous_scale="RdBu",
                      title=f"LIME Explanation for Sample 0 (Target: {class_names[y[0]]})")
    fig_lime.show()


In [ ]:
# 5. Interpretability (Grad-CAM) - Deep Models Only
if model and isinstance(model, nn.Module):
    # Identify target layer (last conv or linear)
    target_layer = next((m for m in reversed(list(model.modules())) if isinstance(m, (nn.Conv1d, nn.Linear))), None)
    if target_layer:
        gc = GradCAM(model, target_layer)
        cam = gc.generate_cam(torch.tensor(X[0]).unsqueeze(0).to(get_device())).cpu().numpy()[0]
        gc.remove_hooks()
        
        # Plot spectrum with CAM importance overlay
        mz_axis = np.array([float(c) for c in feature_names])
        fig_cam = go.Figure()
        fig_cam.add_trace(go.Scatter(x=mz_axis, y=X[0], name="Spectrum", line=dict(color="lightgray", width=1)))
        fig_cam.add_trace(go.Scatter(x=mz_axis, y=X[0], mode="markers", 
                                     marker=dict(color=cam, colorscale="Viridis", size=8, showscale=True), 
                                     name="Importance"))
        fig_cam.update_layout(title="Grad-CAM Importance Map", template="plotly_white", xaxis_title="m/z", yaxis_title="Intensity")
        fig_cam.show()
